In [ ]:
# input data
# Upload student information -> bigquery
# core pipeline (remove loop)
# upload to gcs
# test
# Fastapi

# Input data to BigQuery

In [7]:
from google.cloud import bigquery

from functions.utils.bigquery import DataQuery

# client = bigquery.Client()
mdq = DataQuery()

In [2]:
students_rows = [
    {
        "student_id"         : "stu_p100",
        "preferred_language" : "en",
        "current_status"     : "student",
        "education_level"    : "bachelor",
        "education_major"    : "electrical engineering",
        "target_roles"       : "data science",
        "skills"             : "python;sql;statistics",
        "interests"          : "machine learning;career growth",
        "onboard_grp"        : "job_hunter",
        "onboard_grp_description": "looking to transition into data science role"
    }
]

In [52]:
interactions_rows = [
    {
        "user_id": "stu_p100",
        "feed_id": "TH_F001",
        "ts": "2026-01-06T13:12:10Z",
        "event_type": "view",
        "dwell_ms": 52000
    },
    {
        "user_id": "stu_p100",
        "feed_id": "TH_F001",
        "ts": "2026-01-06T13:13:05Z",
        "event_type": "like",
        "dwell_ms": 0
    },
    {
        "user_id": "stu_p100",
        "feed_id": "TH_F003",
        "ts": "2026-01-05T09:20:11Z",
        "event_type": "view",
        "dwell_ms": 41000
    },
    {
        "user_id": "stu_p100",
        "feed_id": "TH_UNI_043",
        "ts": "2026-01-06T06:40:00Z",
        "event_type": "view",
        "dwell_ms": 12000
    }
]

In [53]:
mdq.upload_data_to_student_table(students_rows)
mdq.upload_data_to_interactions_table(interactions_rows)

Students uploaded successfully
Interactions uploaded successfully


# Core pipeline (remove loop)

#### Import library

In [76]:
from __future__ import annotations

import json
import os
import yaml
import numpy as np
import pandas as pd
import time
import io

from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
from collections import defaultdict
from google.cloud import bigquery
from google.cloud import storage
from google import genai

from functions.utils.logging import get_logger
from functions.utils.config  import PROJECT_ROOT, load_config
from functions.utils.llm_client import build_llm_client_from_yaml
from functions.utils.text_embeddings import GoogleEmbeddingModel
from functions.core.context_builder import build_user_context
from functions.core.history import build_history_summary
from functions.utils.cloudstorage import GoogleCloudStorage

In [79]:
# def ensure_dir(path: str) -> None:
#     """Create directory if it does not exist (idempotent)."""
#     os.makedirs(path, exist_ok=True)
    
def _read_hyde_config(cfg: Dict[str, Any]) -> Tuple[int, int, int, bool, str]:
    """
    Read HyDE-related configuration with safe defaults.

    Returns
    -------
    history_threshold:
        Event count threshold for prompt selection
    recent_k:
        Max number of recent feeds used in HistorySummary
    feed_text_max_chars:
        Per-feed text truncation limit
    include_recent_feeds:
        Whether HistorySummary may include feed snippets
    query_embedding_model_name:
        Embedding model for HyDE queries
    """
    hyde_cfg = cfg.get("hyde", {}) if isinstance(cfg, dict) else {}

    history_threshold = int(hyde_cfg.get("history_threshold", 5))
    recent_k = int(hyde_cfg.get("recent_k", 5))
    feed_text_max_chars = int(hyde_cfg.get("feed_text_max_chars", 240))
    include_recent_feeds = bool(hyde_cfg.get("include_recent_feeds", True))

    # Default to same embedding family as feed embeddings
    query_embedding_model_name = str(
        hyde_cfg.get("query_embedding_model_name")
        or cfg.get("embeddings", {}).get("model_name", "")
        or "gemini-embedding-001"
    )

    # Hard safety guards
    history_threshold = max(1, history_threshold)
    recent_k = max(0, min(recent_k, 10))
    feed_text_max_chars = max(0, min(feed_text_max_chars, 2000))

    return (
        history_threshold,
        recent_k,
        feed_text_max_chars,
        include_recent_feeds,
        query_embedding_model_name,
    )
    
# def read_jsonl(path: str) -> List[Dict[str, Any]]:
#     """
#     Deterministic JSONL reader.

#     Order is preserved, which is critical for any downstream alignment.
#     """
#     rows: List[Dict[str, Any]] = []
#     with open(path, "r", encoding="utf-8") as f:
#         for line_no, line in enumerate(f, start=1):
#             line = line.strip()
#             if not line:
#                 continue
#             try:
#                 rows.append(json.loads(line))
#             except Exception as e:
#                 raise ValueError(f"Invalid JSONL at line {line_no}: {e}") from e
#     return rows

def load_prompts() -> Dict[str, str]:
    """
    Load HyDE prompt templates from parameters/prompts.yaml.
    Expected structure:
      hyde_prompts:
        hyde_a: "..."
        hyde_b: "..."
        hyde_c: "..."
    """
    import yaml
    prompts_path = PROJECT_ROOT / "parameters" / "prompts.yaml"
    with prompts_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}
    return data.get("hyde_prompts", {}) or {}

# =============================================================================
# Prompt selection and rendering
# =============================================================================
def choose_hyde_prompt_key(num_events: int, history_threshold: int = 5) -> str:
    """
    Select HyDE prompt variant based on interaction volume.

    Rules
    -----
    - num_events >= history_threshold → history-heavy (hyde_b)
    - num_events <= 1               → onboarding / sparse (hyde_c)
    - otherwise                     → mixed (hyde_a)
    """
    if num_events >= history_threshold:
        return "hyde_b"
    if num_events <= 1:
        return "hyde_c"
    return "hyde_a"


def render_prompt(
    template: str,
    preferred_language: str,
    user_context_text: str,
    history_summary_text: Optional[str],
) -> str:
    """
    Render a prompt template using strict placeholder substitution.

    Supported placeholders:
    - {{preferred_language}}
    - {{UserContextText}}
    - {{HistorySummaryText}}

    No templating engine is used on purpose to keep behavior explicit.
    """
    s = template.replace("{{preferred_language}}", preferred_language or "th")
    s = s.replace("{{UserContextText}}", user_context_text or "")
    s = s.replace("{{HistorySummaryText}}", history_summary_text or "")
    return s

# =============================================================================
# HyDE output handling
# =============================================================================
def _extract_hyde_query_texts(hyde_json: Dict[str, Any]) -> List[str]:
    """
    Extract query_text values from HyDE JSON output.

    Expected structure:
      {
        "hyde_queries": [
          {"query_id": "...", "query_text": "...", ...},
          ...
        ]
      }

    Order is preserved and MUST match embedding row order.
    """
    if not isinstance(hyde_json, dict):
        raise ValueError("hyde_output must be a dict")

    items = hyde_json.get("hyde_queries") or []
    if not isinstance(items, list):
        raise ValueError("hyde_output.hyde_queries must be a list")

    out: List[str] = []
    for i, it in enumerate(items):
        if not isinstance(it, dict):
            raise ValueError(f"hyde_output.hyde_queries[{i}] must be an object")
        out.append(str(it.get("query_text") or "").strip())

    return out


# def _l2_normalize_rows(x: np.ndarray) -> np.ndarray:
#     """
#     Row-wise L2 normalization.

#     Zero rows are left as zero to avoid NaNs.
#     """
#     if x.ndim != 2:
#         raise ValueError("Expected 2D array for row normalization")

#     norms = np.linalg.norm(x, axis=1, keepdims=True)
#     norms[norms == 0.0] = 1.0
#     return (x / norms).astype(np.float32)


# def _atomic_save_npy(path: str, arr: np.ndarray) -> None:
#     """
#     Best-effort atomic .npy write.

#     Writes to a temp file and renames to avoid partial reads.
#     """
#     tmp = path + ".tmp.npy"
#     np.save(tmp, arr)
#     os.replace(tmp, path)

#### Initail value

In [18]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")
bq = DataQuery()

cfg = load_config()
out_dir = cfg["artifacts"]["user_query_bundles_dir"]
verbose = 1

Bucket exists : hyde-datalake


In [54]:
students = bq.get_students() 
interactions = bq.get_interactions() 
feeds_lookup = bq.get_user_events_json()

/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


#### Read HyDE-related configuration oncee

In [55]:
(history_threshold,recent_k,feed_text_max_chars,include_recent_feeds,query_embedding_model_name) = _read_hyde_config(cfg)
expected_dim = int(cfg.get("embeddings", {}).get("dim", 0) or 0)

In [56]:
prompts = load_prompts()
if not prompts:
    raise ValueError("hyde_prompts missing from parameters/prompts.yaml")
client = build_llm_client_from_yaml(
    parameters_path=str(PROJECT_ROOT / "parameters" / "parameters.yaml"),
    credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
)
# query_embedder = GoogleEmbeddingModel(
#     model_name=query_embedding_model_name,
#     credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
# )
now_iso = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [77]:
# TODO : write this code in core
def embed_texts_gemini(texts:list[str],output_dim:int=768,task_type: str = "RETRIEVAL_DOCUMENT")->np.ndarray:
    if not texts:
        return np.zeros((0,output_dim or 0), dtype=np.float32)
    # step01 : API key
    key = os.getenv("GOOGLE_API_KEY")
    if not key:
        raise ValueError("GOOGLE_API_KEY is not set")
    # step02 : client creation
    client = genai.Client(api_key=key)
    vectors = []
    for raw in texts:
        text = raw.strip() or " "
        resp = client.models.embed_content(
            model="gemini-embedding-001",
            contents=text,
            config={"task_type": task_type},
        )
        if not hasattr(resp, "embeddings") or not resp.embeddings:
            raise RuntimeError("Invalid embedding response")
        values = resp.embeddings[0].values
        vectors.append(values)
    # step03 : ndarray + float32 (friend logic)
    mat = np.asarray(vectors, dtype=np.float32)
    # step04 : optional truncation (before normalize)
    if output_dim is not None:
        if output_dim > mat.shape[1]:
            raise ValueError(
                f"output_dim {output_dim} > embedding dim {mat.shape[1]}"
            )
        mat = mat[:, :output_dim]
    # step05 : L2 normalize (friend logic)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms = np.where(norms == 0.0, 1.0, norms)
    mat = mat / norms

    return mat
# texts = [
#     "Hello, world!",
#     "Vertex AI is great for embeddings.",
#     "I hate everything about you",
#     "21 gungs"
# ]

# vectors = embed_texts_gemini(
#     texts,
#     output_dim=768,
# )

# print(vectors.shape)      # (2, 768)
# print(vectors.dtype)      # float32
# print(np.linalg.norm(vectors[0]))  # ~1.0

(4, 768)
float32
1.0


In [41]:
student_id = "stu_p100"

In [81]:
# -------------------------------------------------
# 1) locate student row
# -------------------------------------------------
student_row_df = students[students["student_id"] == student_id]  # get student that we want from dataframe
if len(student_row_df) == 0:                                     # check there are only one student
    raise ValueError(f"student_id {student_id} not found")
student_row = student_row_df.iloc[0].to_dict()                   # ddataframe -> dict

# -------------------------------------------------
# 2) build context
# -------------------------------------------------
user_ctx = build_user_context(student_row)                       # create user context class
pref_lang = user_ctx.user_context_json.get("preferred_language", "th")

user_events = interactions[interactions["user_id"] == student_id] # get user envent table
num_events = int(len(user_events))

history_summary_text = None

if num_events > 0:
    history_summary_text = build_history_summary(                # build history summary
        user_events,
        preferred_language=pref_lang,
        include_recent_feeds=include_recent_feeds,
        recent_k=recent_k,
        feeds_lookup=feeds_lookup or None,
        feed_text_max_chars=feed_text_max_chars,
    )
    
# -------------------------------------------------
# 3) build prompt
# -------------------------------------------------
prompt_key = choose_hyde_prompt_key(num_events, history_threshold)

template = prompts.get(prompt_key)
if not template:
    raise ValueError(f"Missing prompt '{prompt_key}'")

prompt = render_prompt(
    template=template,
    preferred_language=pref_lang,
    user_context_text=user_ctx.user_context_text,
    history_summary_text=history_summary_text,
)

# -------------------------------------------------
# 4) LLM call
# -------------------------------------------------
hyde_json = client.generate_json(prompt)

# -------------------------------------------------
# 5) Shin embedding
# -------------------------------------------------
hyde_query_texts = _extract_hyde_query_texts(hyde_json)

if hyde_query_texts:
    emb = embed_texts_gemini(
        texts=hyde_query_texts,
        output_dim=768,
        task_type="RETRIEVAL_DOCUMENT",
    )
    if emb.ndim != 2:
        raise ValueError(f"Invalid embedding shape {emb.shape}")
    dim = int(emb.shape[1])
    # print(f"emb.shape -> {emb.shape}")
else:
    dim = expected_dim or 0
    emb = np.zeros((0, dim), dtype=np.float32)
    
# -------------------------------------------------
# 6) save bundle locally
# -------------------------------------------------
now_iso = datetime.now(timezone.utc).isoformat()

bundle = {
    "bundle_version": "v2_hyde_embedded_queries",
    "student_id": student_id,
    "generated_at": now_iso,
    "prompt_key": prompt_key,
    "preferred_language": pref_lang,
    "num_events": num_events,
    "user_context_json": user_ctx.user_context_json,
    "user_context_text": user_ctx.user_context_text,
    "history_summary_text": history_summary_text,
    "hyde_output": hyde_json,
}

if verbose:
    print(bundle)

2026-02-10T02:40:45Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=7.073s | in_tokens=782 | out_tokens=303 | model=gemini-2.5-flash | status=ok


Folder created : gs://<Bucket: hyde-datalake>/stu_p100/embedding/
Folder created : gs://<Bucket: hyde-datalake>/stu_p100/metadata/
Folder created : gs://<Bucket: hyde-datalake>/stu_p100/hyde/
{'student_id': 'stu_p100', 'current_status': 'student', 'education_level': 'bachelor', 'education_major': 'electrical engineering', 'target_roles': 'data science', 'timezone': 'UTC', 'model_name': 'gemini-2.5-flash', 'max_output_tokens': 2048, 'feed_text_max_chars': 240, 'temperature': 0.2}
uploaded JSON -> gs://hyde-datalake/stu_p100/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake/stu_p100/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake/stu_p100/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake/stu_p100/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake/stu_p100/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake/stu_p100/embedding/embedding05.npy
Uploaded text -> gs://hyde-datalake/stu_p100/hyde/hyde_text01.txt
Uploaded text -> gs://hyde-datal

# Upload metadata,hyde and embedding to GCS

In [ ]:
# -------------------------------------------------
# 7) upload to GCS
# -------------------------------------------------
cgs.create_folder(f"{student_id}/embedding/")
cgs.create_folder(f"{student_id}/metadata/")
cgs.create_folder(f"{student_id}/hyde/")

metadata = {
        "student_id":student_id, # 
        "current_status":student_row['current_status'], #
        "education_level":student_row['education_level'], #
        "education_major":student_row['education_major'], #
        "target_roles":student_row['target_roles'], #
        "timezone":cfg["app"]["timezone"], #
        "model_name":cfg["llm"]["model_name"], #
        "max_output_tokens":cfg["llm"]["max_output_tokens"], #
        "feed_text_max_chars":cfg["hyde"]["feed_text_max_chars"], #
        "temperature":cfg["llm"]["temperature"] #
    }
print(metadata)
cgs.upload_json(
    blob_path   = f"{student_id}/metadata/metadata.json",
    json_data   = metadata
)
cgs.upload_npy(
    blob_path   = f"{student_id}/embedding/embedding01.npy",
    array       = emb[0]
)
cgs.upload_npy(
    blob_path   = f"{student_id}/embedding/embedding02.npy",
    array       = emb[1]
)
cgs.upload_npy(
    blob_path   = f"{student_id}/embedding/embedding03.npy",
    array       = emb[2]
)
cgs.upload_npy(
    blob_path   = f"{student_id}/embedding/embedding04.npy",
    array       = emb[3]
)
cgs.upload_npy(
    blob_path   = f"{student_id}/embedding/embedding05.npy",
    array       = emb[4]
)
cgs.upload_text(
    blob_path = f"{student_id}/hyde/hyde_text01.txt",
    text_data = hyde_query_texts[0]
)
cgs.upload_text(
    blob_path = f"{student_id}/hyde/hyde_text02.txt",
    text_data = hyde_query_texts[1]
)
cgs.upload_text(
    blob_path = f"{student_id}/hyde/hyde_text03.txt",
    text_data = hyde_query_texts[2]
)
cgs.upload_text(
    blob_path = f"{student_id}/hyde/hyde_text04.txt",
    text_data = hyde_query_texts[3]
)
cgs.upload_text(
    blob_path = f"{student_id}/hyde/hyde_text05.txt",
    text_data = hyde_query_texts[4]
)

# Test

In [85]:
cgs.read_text("stu_p100/hyde/hyde_text01.txt")

'data science career path for electrical engineering students'

# FastAPI